# SupportIQ Intent Classifier Training
This notebook handles data preprocessing, model training, evaluation, and export.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    # Normalize phone numbers: +92, 03xx etc
    phone_pattern = r'(\+?92|0)?\s*3\d{2}[-\s]?\d{3}[-\s]?\d{4}'
    text = re.sub(phone_pattern, '__PHONE__', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
# Load dataset
df = pd.read_csv('backend/data/intent_dataset.csv')
df['text'] = df['text'].apply(preprocess_text)
X = df['text']
y = df['intent']

print(f"Dataset loaded: {len(df)} rows")
print(df['intent'].value_counts())

In [ ]:
# Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train set: {len(X_train)}, Test set: {len(X_test)}")

In [ ]:
# Build Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_df=0.95, min_df=2)),
    ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=1.0, random_state=42, dual=False), method='sigmoid'))
])

pipeline.fit(X_train, y_train)
print("Model trained successfully")

In [ ]:
# Evaluation
y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=pipeline.classes_, 
            yticklabels=pipeline.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Export model
model_dir = 'backend/models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
joblib.dump(pipeline, os.path.join(model_dir, 'classifier.pkl'))
print(f"Model saved to {model_dir}/classifier.pkl")

In [ ]:
# Test predictions
test_queries = [
    'DHA mein plot ka rate kya hai',
    'mujhe buy karna hai 0312-1234567',
    'manager se baat karni hai abhi',
    'what is the maintenance fee',
    'I am ready to invest 2 crore'
]

for query in test_queries:
    processed = preprocess_text(query)
    proba = pipeline.predict_proba([processed])[0]
    max_idx = np.argmax(proba)
    intent = pipeline.classes_[max_idx]
    confidence = proba[max_idx]
    print(f"Query: {query}\nIntent: {intent}, Confidence: {confidence:.4f}\n{'-'*30}")